# 06 - The App: Gradio + Hugging Face Spaces

Packaging the final system into a web app and deploying it for free.

**What the app uses** (the best setup from the evaluation):
- fine-tuned bge-small + BM25 hybrid search (judged Hit@5 0.98, millisecond latency on CPU)
- company / year filters
- Gemini for grounded, cited answers, with chart images sent alongside the text

Runtime: CPU is fine for this notebook.

## Setup

In [1]:
# Installing libraries (-U so sentence-transformers matches the version the model was saved with in Notebook 5)
# After this cell finishes: Runtime -> Restart session, then continue from the next cell
!pip install -q -U sentence-transformers
!pip install -q gradio rank_bm25 google-genai huggingface_hub chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 17.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60

In [2]:
# Mounting Drive and setting paths
from google.colab import drive
drive.mount('/content/drive')

import json, shutil, time, os
from pathlib import Path
import numpy as np
import chromadb

PROJECT_DIR = Path("/content/drive/MyDrive/annual-report-rag")
MODEL_DIR = PROJECT_DIR / "models" / "bge-small-annual-reports"
APP = Path("/content/app")
shutil.rmtree(APP, ignore_errors=True)
(APP / "data" / "images").mkdir(parents=True)
print("Model found:", MODEL_DIR.exists())

# checking the library version (the model was saved with sentence-transformers 6.x in Notebook 5)
import sentence_transformers
v = sentence_transformers.__version__
print("sentence-transformers:", v)
assert int(v.split(".")[0]) >= 6, "Old version loaded: Runtime -> Restart session, then rerun from this cell"

Mounted at /content/drive
Model found: True
sentence-transformers: 6.1.0


## Packaging the data

In [3]:
# Exporting the fine-tuned index from Chroma into simple files the app can load fast
CHROMA_LOCAL = f"/content/chroma_{int(time.time())}"
shutil.copytree(PROJECT_DIR / "chroma", CHROMA_LOCAL)
col = chromadb.PersistentClient(path=CHROMA_LOCAL).get_collection("annual_reports_ft")
got = col.get(include=["embeddings", "documents"])
ids = got["ids"]
E = np.asarray(got["embeddings"], dtype=np.float32)
np.save(APP / "data" / "embeddings.npy", E)
(APP / "data" / "ids.json").write_text(json.dumps(ids))
(APP / "data" / "doc_text.json").write_text(json.dumps(dict(zip(ids, got["documents"])), ensure_ascii=False))
print("embeddings:", E.shape)

embeddings: (4059, 384)


In [4]:
# Slimming the docstore and copying only the chart images the app shows
docstore = json.loads((PROJECT_DIR / "docstore.json").read_text())
KEEP = ["id", "type", "company", "year", "doc_id", "page", "content", "summary", "caption"]
slim, copied = {}, 0
for i in ids:
    d = docstore[i]
    s = {k: d.get(k) for k in KEEP}
    if d["type"] == "image" and d.get("image_path") and Path(d["image_path"]).exists():
        name = Path(d["image_path"]).name
        shutil.copy(d["image_path"], APP / "data" / "images" / name)
        s["image"] = name
        copied += 1
    slim[i] = s
(APP / "data" / "docstore.json").write_text(json.dumps(slim, ensure_ascii=False))
print(f"{len(slim)} items, {copied} chart images copied")

4059 items, 130 chart images copied


In [5]:
# Copying the fine-tuned embedding model and checking the app size
shutil.copytree(MODEL_DIR, APP / "model")
total = sum(f.stat().st_size for f in APP.rglob("*") if f.is_file())
print(f"App folder size: {total / 1e6:.0f} MB")
for p in sorted(APP.rglob("*"))[:15]:
    print("  ", p.relative_to(APP))

App folder size: 161 MB
   data
   data/doc_text.json
   data/docstore.json
   data/embeddings.npy
   data/ids.json
   data/images
   data/images/au_small_finance_bank_2023_image_133.png
   data/images/au_small_finance_bank_2023_image_149.png
   data/images/au_small_finance_bank_2023_image_166.png
   data/images/au_small_finance_bank_2023_image_196.png
   data/images/au_small_finance_bank_2023_image_197.png
   data/images/au_small_finance_bank_2023_image_199.png
   data/images/au_small_finance_bank_2023_image_202.png
   data/images/au_small_finance_bank_2023_image_218.png
   data/images/au_small_finance_bank_2023_image_277.png


## Writing the app

In [6]:
%%writefile /content/app/app.py
# Annual Report Analyst: multimodal RAG over company annual reports
import os, json, re, time
from pathlib import Path
import numpy as np
import gradio as gr
from PIL import Image
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi
from google import genai
from google.genai import types

HERE = Path(__file__).parent
DATA = HERE / "data"

# loading the index (fine-tuned embeddings, original content, chart images)
docstore = json.loads((DATA / "docstore.json").read_text())
ids = json.loads((DATA / "ids.json").read_text())
doc_text = json.loads((DATA / "doc_text.json").read_text())
M = np.load(DATA / "embeddings.npy")
embedder = SentenceTransformer(str(HERE / "model"), device="cpu")

# building keyword search
TOKEN = re.compile(r"[a-z0-9]+(?:[.,][0-9]+)*")
def tok(s): return TOKEN.findall(s.lower())
bm25 = BM25Okapi([tok(doc_text[i]) for i in ids])
comp_arr = np.array([docstore[i]["company"] for i in ids])
year_arr = np.array([docstore[i]["year"] for i in ids])

COMPANIES = sorted(set(comp_arr.tolist()))
YEARS = sorted(set(year_arr.tolist()))
ALIASES = {c: [c.replace("_", " ")] for c in COMPANIES}
ALIASES.setdefault("au_small_finance_bank", []).extend(["au bank", "au sfb"])
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "

def pretty_company(c): return c.replace("_", " ").title()
def fy_label(y): return f"FY {int(y) - 1}-{y[2:]}"

# parsing company and fiscal year from the question
def find_years(q):
    ys, q = set(), q.lower()
    for m in re.finditer(r"(20\d{2})\s*[-–/]\s*(\d{2,4})", q):
        end = m.group(2); ys.add(end if len(end) == 4 else "20" + end)
    q = re.sub(r"(20\d{2})\s*[-–/]\s*(\d{2,4})", " ", q)
    for m in re.finditer(r"fy\s*'?(\d{2,4})\b", q):
        y = m.group(1); ys.add(y if len(y) == 4 else "20" + y)
    q = re.sub(r"fy\s*'?(\d{2,4})\b", " ", q)
    ys.update(re.findall(r"\b(20\d{2})\b", q))
    return sorted(y for y in ys if y in YEARS)

def parse_query(q):
    ql = q.lower()
    company = next((c for c, names in ALIASES.items() if any(n in ql for n in names)), None)
    return company, find_years(q)

# hybrid search: fine-tuned dense + BM25, fused with reciprocal rank fusion
def filter_mask(company, years):
    m = np.ones(len(ids), bool)
    if company: m &= comp_arr == company
    if years:   m &= np.isin(year_arr, years)
    return m

def ranked(scores, mask, n):
    s = np.where(mask, scores, -np.inf)
    top = np.argpartition(-s, min(n, len(s) - 1))[:n]
    return [ids[j] for j in top[np.argsort(-s[top])] if np.isfinite(s[j])]

def rrf(*lists, k=60):
    s = {}
    for L in lists:
        for r, i in enumerate(L, 1):
            s[i] = s.get(i, 0) + 1 / (k + r)
    return [i for i, _ in sorted(s.items(), key=lambda x: -x[1])]

def search(q, company, years, top=8):
    mask = filter_mask(company, years)
    qv = embedder.encode([QUERY_PREFIX + q], normalize_embeddings=True)[0]
    return rrf(ranked(M @ qv, mask, 30), ranked(bm25.get_scores(tok(q)), mask, 30))[:top]

# answering with Gemini, falling back to other models when one is busy
ANSWER_PROMPT = """You are a financial analyst assistant answering questions about company annual reports.
Rules:
- Use ONLY the sources provided below. Never use outside knowledge.
- Only report numbers that appear in the sources. You MAY compute a simple difference or
  % change between two figures stated in the sources, and must show the calculation.
  NEVER derive new quantities (e.g. back-calculating a total from a percentage).
- Quote exact numbers with their units (₹ crore, %, etc.) and fiscal year.
- Cite every fact like [AU Small Finance Bank FY2024, p.148].
- Include only points that directly answer the question; leave out tangential data.
- If the sources don't contain the answer, reply: "I couldn't find this in the provided reports."
- Keep it concise: a direct answer first, then 1-3 supporting points."""

ANSWER_MODELS = ["gemini-3.5-flash-lite", "gemini-3.8-flash", "gemini-3.7-flash", "gemini-3.6-flash"]
API_KEY = os.environ.get("GEMINI_API_KEY")
client = genai.Client(api_key=API_KEY, http_options=types.HttpOptions(timeout=120_000)) if API_KEY else None  # 120 s max per call

RETRYABLE = ("500", "502", "503", "504", "UNAVAILABLE", "DEADLINE_EXCEEDED", "INTERNAL", "timed out", "timeout")

def generate(contents):
    for model in ANSWER_MODELS:
        for attempt in range(2):
            t = time.time()
            try:
                r = client.models.generate_content(model=model, contents=contents,
                        config=types.GenerateContentConfig(temperature=0))
                print(f"[answer] {model} ok in {time.time() - t:.1f}s")
                return r.text, model
            except Exception as ex:
                msg = str(ex)
                print(f"[answer] {model} attempt {attempt + 1} failed after {time.time() - t:.1f}s: {msg[:150]}")
                if "429" in msg or "RESOURCE_EXHAUSTED" in msg:
                    break                                   # out of quota: next model
                if any(code in msg for code in RETRYABLE):
                    time.sleep(3)                           # temporary server problem: retry, then next model
                    continue
                raise                                       # real bug: show it
    return "All models are busy right now, please try again in a minute.", "none"

def cite(d):
    return f"[{pretty_company(d['company'])} FY{d['year']}, p.{d['page']}]"

def build_contents(q, hits):
    contents = [ANSWER_PROMPT, f"QUESTION: {q}", "SOURCES:"]
    for k, i in enumerate(hits, 1):
        d = docstore[i]
        if d["type"] == "image" and d.get("image"):
            contents += [f"Source {k} {cite(d)}: chart. Summary: {d.get('summary', '')}",
                         Image.open(DATA / "images" / d["image"])]
        elif d["type"] == "table":
            contents.append(f"Source {k} {cite(d)}: table\n{d['content'][:6000]}")
        else:
            contents.append(f"Source {k} {cite(d)}: text\n{d['content']}")
    return contents

COMPANY_CHOICES = {"Auto-detect": None, **{pretty_company(c): c for c in COMPANIES}}
YEAR_CHOICES = {"Auto-detect": None, **{fy_label(y): y for y in YEARS}}

def ask(question, company_choice="Auto-detect", year_choice="Auto-detect"):
    if not question or not question.strip():
        return "Please type a question.", "", []
    if client is None:
        return "GEMINI_API_KEY is not set. Add it as a secret to use the app.", "", []
    company, years = parse_query(question)
    if COMPANY_CHOICES.get(company_choice): company = COMPANY_CHOICES[company_choice]
    if YEAR_CHOICES.get(year_choice):       years = [YEAR_CHOICES[year_choice]]

    t0 = time.time()
    hits = search(question, company, years)
    t_ret = time.time() - t0
    text, model = generate(build_contents(question, hits))
    t_ans = time.time() - t0 - t_ret

    lines = []
    for k, i in enumerate(hits, 1):
        d = docstore[i]
        preview = (d.get("summary") or d["content"]).replace("\n", " ")[:160]
        lines.append(f"**{k}.** {cite(d)} · *{d['type']}* · {preview}…")
    filt = f"{pretty_company(company) if company else 'any company'}, " \
           f"{', '.join(fy_label(y) for y in years) if years else 'any year'}"
    sources_md = "\n\n".join(lines) + \
        f"\n\n<sub>Filters: {filt} · retrieval {t_ret:.2f}s · answer {t_ans:.1f}s by {model}</sub>"
    gallery = [(str(DATA / "images" / docstore[i]["image"]), cite(docstore[i]))
               for i in hits if docstore[i]["type"] == "image" and docstore[i].get("image")]
    return text, sources_md, gallery

EXAMPLES = [
    "What was the gross NPA closing balance in FY 2023-24?",
    "How did total advances change between FY23 and FY24?",
    "What are the key risks highlighted by management in FY2025?",
    "Show me the business segments of AU Small Finance Bank in FY2023",
    "What was the total deposit amount in FY 2024-25?",
]

with gr.Blocks(title="Annual Report Analyst", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📊 Annual Report Analyst\n"
                "Ask questions about company annual reports. Answers come only from the reports, "
                "with page citations, and draw on text, tables and charts.")
    with gr.Row():
        q = gr.Textbox(label="Your question", placeholder="e.g. What was net profit in FY 2024-25?", scale=4)
        btn = gr.Button("Ask", variant="primary", scale=1)
    with gr.Row():
        comp = gr.Dropdown(list(COMPANY_CHOICES), value="Auto-detect", label="Company")
        yr = gr.Dropdown(list(YEAR_CHOICES), value="Auto-detect", label="Report year")
    ans = gr.Markdown()
    with gr.Accordion("Sources", open=True):
        src = gr.Markdown()
        gal = gr.Gallery(label="Charts used", columns=3, height="auto")
    gr.Examples(EXAMPLES, inputs=q)
    btn.click(ask, [q, comp, yr], [ans, src, gal])
    q.submit(ask, [q, comp, yr], [ans, src, gal])

if __name__ == "__main__":
    demo.launch()

Writing /content/app/app.py


In [7]:
# Writing requirements.txt and the Hugging Face Space README (with pinned versions)
from importlib.metadata import version
import gradio

reqs = [f"{p}=={version(p)}" for p in
        ["sentence-transformers", "rank-bm25", "google-genai", "numpy", "pillow"]]
(APP / "requirements.txt").write_text("\n".join(reqs) + "\n")

(APP / "README.md").write_text(f"""---
title: Annual Report Analyst
emoji: 📊
colorFrom: blue
colorTo: indigo
sdk: gradio
sdk_version: {gradio.__version__}
app_file: app.py
pinned: false
---

# Annual Report Analyst

Multimodal RAG over company annual reports: ask a question, get a grounded answer with page citations,
drawing on text, tables and charts.

- Retrieval: fine-tuned bge-small-en-v1.5 + BM25, fused with reciprocal rank fusion, with company/year filters
- Generation: Gemini, sources only, every fact cited
""")
print((APP / "requirements.txt").read_text())

sentence-transformers==6.1.0
rank-bm25==0.2.2
google-genai==2.12.1
numpy==2.1.3
pillow==11.3.0



## Testing in Colab

In [14]:
# Loading the app with the Gemini key from Colab secrets
import sys, importlib
from google.colab import userdata
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
sys.path.insert(0, str(APP))
import app
importlib.reload(app)

# asking one question without the UI
answer, sources, charts = app.ask("What was the gross NPA closing balance in FY 2023-24?")
print(answer, "\n")
print(sources)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/content/app/app.py:173: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="Annual Report Analyst", theme=gr.themes.Soft()) as demo:


[answer] gemini-3.5-flash-lite ok in 1.1s
The gross NPA closing balance (Total Non-Performing Advances) for FY 2023-24 (as of March 31, 2024) was **₹ 1,237.40 Crore** [AU Small Finance Bank FY2024, p.148].

* **Sub-standard:** ₹ 844.53 Crore [AU Small Finance Bank FY2024, p.148]
* **Doubtful:** ₹ 387.77 Crore [AU Small Finance Bank FY2024, p.148]
* **Loss:** ₹ 5.10 Crore [AU Small Finance Bank FY2024, p.148] 

**1.** [Au Small Finance Bank FY2024, p.148] · *table* · This table presents the movement of gross standard advances, NPAs, and provisions for AU Small Finance Bank for the prior year ended March 31, 2023. Total gross…

**2.** [Au Small Finance Bank FY2024, p.149] · *table* · This table summarizes key asset quality ratios for AU Small Finance Bank as of March 31, 2024, compared to March 31, 2023. The Gross NPA to Gross Advances ratio…

**3.** [Au Small Finance Bank FY2024, p.148] · *table* · This table presents the movement of gross standard advances, non-performing advances (NPA

In [ ]:
# Launching the app with a temporary public link (stop this cell when done testing)
app.demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://9e4cb53d41308d2f3d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[answer] gemini-3.5-flash-lite ok in 1.0s
[answer] gemini-3.5-flash-lite ok in 1.0s
[answer] gemini-3.5-flash-lite ok in 1.0s
[answer] gemini-3.5-flash-lite ok in 1.4s
[answer] gemini-3.5-flash-lite ok in 2.8s
